# SPY Daily Volatility Modelling

End-to-end pipeline for one-step-ahead realised-variance forecasting on SPY
(daily frequency). Five models are evaluated on Random Forecast Periods (RFP):

| Section | Model |
|--------:|:------|
| 4 | GJR-GARCH / GARCH-X |
| 5 | MS-GARCH |
| 6 | LSTM with Attention |
| 7 | Transformer |
| 8 | XGBoost |

Each model loads its previously-tuned best hyper-parameters from
`<model>/<model>_validation_results.csv`. Set `tune_<model> = True` in the
config cell to re-run grid search instead.


## Environment Requirements

To run this notebook successfully, ensure your environment matches the following specifications from `environment.yaml`:

```yaml
name: tsa-project
channels:
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - numpy
  - pandas
  - matplotlib
  - scikit-learn
  - tqdm
  - scipy
  - pytorch::pytorch
  - ipykernel
  - pip:
      - statsmodels
      - arch
      - xgboost
      - yfinance
      - rpy2
```


## 0. Configuration

In [1]:
# Targets and frequency
TARGET = "SPY"
FREQ = "daily"
EXOG_VARIANTS = ["no_exog", "with_exog"]

# Re-tune flags - default False uses saved best params from <model>/<model>_validation_results.csv
tune_garch = False
tune_msgarch = False
tune_lstm_attention = False
tune_transformer = False
tune_xgboost = False

# Common run knobs
SEED = 42
SHOW_PLOTS = True


In [2]:
# Output directories. Keys here must match the keys used by
# load_best_params() and save_rfp_artifacts() further down.
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DAILY_DIR = DATA_DIR / "daily"
SPLITS_DIR = DATA_DIR / "splits"

MODEL_DIRS = {
    "garch": ROOT / "GARCH",
    "msgarch": ROOT / "MSGARCH",
    "lstm_attention": ROOT / "lstm_attention",
    "transformer": ROOT / "transformer",
    "xgboost": ROOT / "xgboost",
}
for d in [DATA_DIR, DAILY_DIR, SPLITS_DIR, *MODEL_DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)
print("Output directories ready.")


Output directories ready.


## 1. Imports

In [30]:
from __future__ import annotations
import itertools
import math
import random
import warnings
from dataclasses import dataclass, asdict, field
from statistics import NormalDist
from typing import Iterator, Optional, Iterable
import os

import yfinance as yf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from arch import arch_model

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

import xgboost as xgb



warnings.filterwarnings("ignore")
np.random.seed(SEED)
random.seed(SEED)

VAR_LEVELS = (0.01, 0.05)
# ENDO_COLS lists every column that is *not* an exogenous regressor —
# used by exog_columns(df) to identify true exogenous features.
ENDO_COLS = {"ret", "date", "ret_lag1", "ret_sq_lag1", "neg_ret_sq_lag1",
             "RV_5_lag1", "RV_10_lag1", "RV_22_lag1"}


## 2. Download Data

Pulls SPY plus exogenous market series (OIL, GOLD, VIX, DXY) from Yahoo Finance
into `data/daily/`. The exogenous series are needed to build the `with_exog`
split. Skip-if-exists keeps re-runs cheap.


In [ ]:
TICKERS = {
    "SPY": "SPY",
    "OIL": "CL=F",
    "GOLD": "GC=F",
    "VIX": "^VIX",
    "DXY": "DX-Y.NYB",
}
START = "2000-01-01"

def download_daily(name: str, ticker: str, force: bool = False) -> pd.DataFrame:
    path = DAILY_DIR / f"{name}_daily.csv"
    if path.exists() and not force:
        return pd.read_csv(path, skiprows=[1, 2], header=0).rename(columns={"Price": "Date"})
    df = yf.download(ticker, start=START, interval="1d", auto_adjust=True, progress=False)
    df.to_csv(path)
    return df

for name, ticker in TICKERS.items():
    df = download_daily(name, ticker)
    print(f"{name:5s} | {ticker:>10s} | rows={len(df):>6,}")


SPY   |        SPY | rows= 6,622
OIL   |       CL=F | rows= 6,450
GOLD  |       GC=F | rows= 6,441
VIX   |       ^VIX | rows= 6,622
DXY   |   DX-Y.NYB | rows= 6,651


## 3. Build Splits

Builds train / val / test CSVs under `data/splits/daily/{no_exog,with_exog}/SPY/`
together with the RFP window definitions in `data/splits/rfp/daily_windows.csv`.

Manifest periods (matches `data/splits/manifest.yaml`):

| stage | window |
|:------|:-------|
| train | 2004-01-01 → 2021-12-31 |
| val   | 2022-01-01 → 2023-12-31 |
| test  | 2024-01-01 → 2026-04-29 |

RFP regimes for `daily`: GFC, OIL_CRASH, COVID, ENERGY_22, CALM_17_19
(60-day windows, up to 5 per regime).


In [5]:
SAMPLE_START = pd.Timestamp("2004-01-01")
SPLIT_RANGES = {
    "train": (pd.Timestamp("2004-01-01"), pd.Timestamp("2021-12-31")),
    "val":   (pd.Timestamp("2022-01-01"), pd.Timestamp("2023-12-31")),
    "test":  (pd.Timestamp("2024-01-01"), pd.Timestamp("2026-04-29")),
}
RFP_REGIMES = {
    "GFC":        (pd.Timestamp("2007-07-01"), pd.Timestamp("2009-06-30")),
    "OIL_CRASH":  (pd.Timestamp("2014-06-01"), pd.Timestamp("2016-02-29")),
    "COVID":      (pd.Timestamp("2020-02-01"), pd.Timestamp("2020-12-31")),
    "ENERGY_22":  (pd.Timestamp("2022-01-01"), pd.Timestamp("2023-06-30")),
    "CALM_17_19": (pd.Timestamp("2017-01-01"), pd.Timestamp("2019-12-31")),
}
RFP_WINDOW_LEN = 60
RFP_N_PER_REGIME = 5
RFP_EMBARGO = 1
RV_WINDOWS = (5, 10, 22)
SERIES = ["SPY", "OIL", "GOLD", "VIX", "DXY"]


def load_close(name: str) -> pd.Series:
    path = DAILY_DIR / f"{name}_daily.csv"
    df = pd.read_csv(path, skiprows=[1, 2], header=0).rename(columns={"Price": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    s = df.set_index("Date")["Close"].astype(float).sort_index()
    return s


def build_panel() -> pd.DataFrame:
    closes = {s: load_close(s) for s in SERIES}
    panel = pd.concat(closes, axis=1).dropna(how="any")
    panel = panel.loc[panel.index >= SAMPLE_START]
    panel.index.name = "date"
    return panel


def build_features(panel: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=panel.index)
    for s in ["SPY", "OIL", "GOLD", "DXY"]:
        ret = np.log(panel[s]).diff()
        out[f"{s}_ret"] = ret
        out[f"{s}_ret_lag1"] = ret.shift(1)
        if s != "DXY":
            ret_sq = ret ** 2
            out[f"{s}_ret_sq_lag1"] = ret_sq.shift(1)
            out[f"{s}_neg_ret_sq_lag1"] = ((ret < 0) * ret_sq).shift(1)
            for w in RV_WINDOWS:
                out[f"{s}_RV_{w}_lag1"] = ret.rolling(w).std().shift(1)
    out["vix_log"] = np.log(panel["VIX"])
    out["vix_log_lag1"] = out["vix_log"].shift(1)
    out["vix_log_diff_lag1"] = out["vix_log"].diff().shift(1)
    return out.dropna()


def build_cell(features: pd.DataFrame, target: str, use_exog: bool) -> pd.DataFrame:
    cols = {"ret": features[f"{target}_ret"]}
    cols["ret_lag1"]         = features[f"{target}_ret_lag1"]
    cols["ret_sq_lag1"]      = features[f"{target}_ret_sq_lag1"]
    cols["neg_ret_sq_lag1"]  = features[f"{target}_neg_ret_sq_lag1"]
    for w in RV_WINDOWS:
        cols[f"RV_{w}_lag1"] = features[f"{target}_RV_{w}_lag1"]
    if use_exog:
        for s in ["SPY", "OIL", "GOLD"]:
            if s == target:
                continue
            cols[f"{s}_ret_lag1"] = features[f"{s}_ret_lag1"]
        cols["DXY_ret_lag1"]      = features["DXY_ret_lag1"]
        cols["vix_log_lag1"]      = features["vix_log_lag1"]
        cols["vix_log_diff_lag1"] = features["vix_log_diff_lag1"]
    df = pd.DataFrame(cols, index=features.index).dropna()
    df.index.name = "date"
    return df.reset_index()


def write_splits() -> pd.DataFrame:
    panel = build_panel()
    features = build_features(panel)
    rows = []
    for use_exog, sub in [(False, "no_exog"), (True, "with_exog")]:
        cell_df = build_cell(features, TARGET, use_exog)
        for stage, (s_start, s_end) in SPLIT_RANGES.items():
            slc = cell_df[(cell_df["date"] >= s_start) & (cell_df["date"] <= s_end)].reset_index(drop=True)
            outdir = SPLITS_DIR / FREQ / sub / TARGET
            outdir.mkdir(parents=True, exist_ok=True)
            slc.to_csv(outdir / f"{stage}.csv", index=False, date_format="%Y-%m-%d")
            rows.append({"sub": sub, "stage": stage, "rows": len(slc), "cols": slc.shape[1]})
    return pd.DataFrame(rows), features


def generate_rfp_windows(features: pd.DataFrame) -> pd.DataFrame:
    rng = random.Random(SEED)
    test_start = SPLIT_RANGES["test"][0]
    dates = pd.DatetimeIndex(features.index)
    rows = []
    for regime_name, (rs, re_) in RFP_REGIMES.items():
        if re_ >= test_start:
            print(f"skip regime {regime_name}: span end {re_.date()} >= test_start {test_start.date()}")
            continue
        in_regime = [dates.get_loc(d) for d in dates[(dates >= rs) & (dates <= re_)]]
        candidates = []
        for pos in in_regime:
            if pos + RFP_WINDOW_LEN - 1 >= len(dates):
                continue
            if pos - RFP_EMBARGO - 1 < 0:
                continue
            forecast_end = dates[pos + RFP_WINDOW_LEN - 1]
            if forecast_end >= test_start:
                continue
            candidates.append(pos)
        rng.shuffle(candidates)
        chosen = []
        for c in candidates:
            if all(abs(c - x) >= RFP_WINDOW_LEN for x in chosen):
                chosen.append(c)
            if len(chosen) == RFP_N_PER_REGIME:
                break
        chosen.sort()
        for k, start_pos in enumerate(chosen, start=1):
            rows.append({
                "window_id": f"d_{regime_name}_{k}",
                "regime": regime_name,
                "fit_end": dates[start_pos - RFP_EMBARGO - 1].date().isoformat(),
                "forecast_start": dates[start_pos].date().isoformat(),
                "forecast_end": dates[start_pos + RFP_WINDOW_LEN - 1].date().isoformat(),
            })
    return pd.DataFrame(rows)


summary, _features = write_splits()
print("Splits written under", SPLITS_DIR / FREQ)
display(summary)

(SPLITS_DIR / "rfp").mkdir(parents=True, exist_ok=True)
rfp_df = generate_rfp_windows(_features)
rfp_path = SPLITS_DIR / "rfp" / "daily_windows.csv"
rfp_df.to_csv(rfp_path, index=False)
print(f"RFP windows: {len(rfp_df)} -> {rfp_path}")
display(rfp_df.head())


Splits written under /Users/harsh/Gatech_courses/TSA/project/data/splits/daily


,sub,stage,rows,cols
0,no_exog,train,4472,8
1,no_exog,val,501,8
2,no_exog,test,583,8
3,with_exog,train,4472,13
4,with_exog,val,501,13
5,with_exog,test,583,13


RFP windows: 22 -> /Users/harsh/Gatech_courses/TSA/project/data/splits/rfp/daily_windows.csv


,window_id,regime,fit_end,forecast_start,forecast_end
0,d_GFC_1,GFC,2007-09-21,2007-09-25,2007-12-18
1,d_GFC_2,GFC,2008-01-07,2008-01-09,2008-04-04
2,d_GFC_3,GFC,2008-04-24,2008-04-28,2008-07-22
3,d_GFC_4,GFC,2008-10-17,2008-10-21,2009-01-15
4,d_GFC_5,GFC,2009-02-26,2009-03-02,2009-05-26


## 4. Reusable Utilities

Shared helpers used by every model section: split loaders, the RFP window
iterator, the metric set, and the per-model summary printer (`display_rfp`).


In [7]:
def load_cell(target: str = TARGET, freq: str = FREQ,
              exog: str = "no_exog") -> dict[str, pd.DataFrame]:
    """Load train/val/test for one (target, freq, exog) cell."""
    base = SPLITS_DIR / freq / exog / target
    frames = {}
    for stage in ("train", "val", "test"):
        df = pd.read_csv(base / f"{stage}.csv", parse_dates=["date"]).sort_values("date")
        frames[stage] = df.reset_index(drop=True)
    return frames


def feature_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in {"date", "ret"}]


def exog_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in ENDO_COLS]


def realized_variance(ret: pd.Series) -> np.ndarray:
    return np.square(ret.to_numpy(dtype=np.float64) * 100.0)


def metrics(y_true: np.ndarray, pred_var: np.ndarray, returns_pct: np.ndarray) -> dict:
    pred_var = np.maximum(pred_var, 1e-8)
    err = pred_var - y_true
    out = {
        "mse": float(np.mean(err ** 2)),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mae": float(np.mean(np.abs(err))),
        "qlike": float(np.mean(np.log(pred_var) + y_true / pred_var)),
    }
    sigma = np.sqrt(pred_var)
    for level in VAR_LEVELS:
        z = NormalDist().inv_cdf(level)
        hits = returns_pct < z * sigma
        out[f"var_{int(level * 100)}_hit_rate"] = float(np.mean(hits))
    return out


def add_residual_columns(fc: pd.DataFrame) -> pd.DataFrame:
    fc = fc.copy()
    fc["std_resid"] = fc["ret_pct"] / np.maximum(fc["pred_vol"], 1e-8)
    fc["squared_std_resid"] = fc["std_resid"] ** 2
    return fc


In [8]:
@dataclass
class RFPWindow:
    window_id: str
    regime: str
    target: str
    use_exog: bool
    fit_end: pd.Timestamp
    forecast_start: pd.Timestamp
    forecast_end: pd.Timestamp
    train: pd.DataFrame = field(repr=False)
    forecast: pd.DataFrame = field(repr=False)

    @property
    def n_train(self) -> int:
        return len(self.train)

    @property
    def n_forecast(self) -> int:
        return len(self.forecast)


def iter_rfp_windows(
    target: str = TARGET, freq: str = FREQ, use_exog: bool = False,
    regimes: Optional[Iterable[str]] = None,
) -> Iterator[RFPWindow]:
    sub = "with_exog" if use_exog else "no_exog"
    base = SPLITS_DIR / freq / sub / target
    train = pd.read_csv(base / "train.csv", parse_dates=["date"])
    val = pd.read_csv(base / "val.csv", parse_dates=["date"])
    data = pd.concat([train, val], ignore_index=True).sort_values("date").reset_index(drop=True)
    windows = pd.read_csv(
        SPLITS_DIR / "rfp" / f"{freq}_windows.csv",
        parse_dates=["fit_end", "forecast_start", "forecast_end"],
    )
    if regimes is not None:
        windows = windows[windows["regime"].isin(set(regimes))]
    for row in windows.itertuples(index=False):
        train_slice = data[data["date"] <= row.fit_end].reset_index(drop=True)
        forecast_slice = data[
            (data["date"] >= row.forecast_start) & (data["date"] <= row.forecast_end)
        ].reset_index(drop=True)
        yield RFPWindow(
            window_id=row.window_id, regime=row.regime,
            target=target, use_exog=use_exog,
            fit_end=row.fit_end,
            forecast_start=row.forecast_start,
            forecast_end=row.forecast_end,
            train=train_slice, forecast=forecast_slice,
        )


In [9]:
def summarise_rfp(results_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-window RFP results into the canonical (exog x regime)
    QLIKE/RMSE table used in display_rfp."""
    if results_df.empty:
        return results_df
    g = results_df.groupby(["exog", "regime"])[["qlike", "rmse"]].mean().round(4)
    return g.reset_index()


def display_rfp(results_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """Print and return the QLIKE/RMSE summary for one model's RFP results."""
    print(f"\n{'=' * 60}\nRFP results — {model_name}\n{'=' * 60}")
    if results_df.empty:
        print("(no windows evaluated)")
        return results_df
    overall = (
        results_df.groupby("exog")[["qlike", "rmse"]].mean()
        .round(4).rename_axis("exog").reset_index()
    )
    summary = summarise_rfp(results_df)
    print("\nMean across all windows (per exog):")
    display(overall)
    print("\nMean by regime:")
    display(summary.pivot(index="regime", columns="exog", values=["qlike", "rmse"]))
    return summary


def save_rfp_artifacts(model_name: str, results_df: pd.DataFrame,
                       forecasts: dict[str, pd.DataFrame]) -> None:
    out = MODEL_DIRS[model_name]
    rfp_dir = out / "rfp"
    fc_dir = rfp_dir / "forecasts"
    fc_dir.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(rfp_dir / f"{model_name}_rfp_results.csv", index=False)
    for fname, fc in forecasts.items():
        fc.to_csv(fc_dir / f"{fname}.csv", index=False)
    print(f"Saved {len(results_df)} window results -> {rfp_dir.relative_to(ROOT)}")


# --- INJECTED DATA (FILTERED FOR SPY/DAILY ONLY) ---
PRELOADED_CSVS = {"garch_grid_search_results.csv": {"target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"], "best_p": [1, 1], "best_o": [1, 1], "best_q": [1, 1], "aic": [11126.927904781343, 11100.087289963987]}, "garch_validation_results.csv": {"mse": [7.277116214706341, 7.414484089789892], "rmse": [2.6976130587440337, 2.722955028969426], "mae": [1.54160667933971, 1.5872817262657362], "qlike": [1.244544592880766, 1.4135842986830192], "var_1_hit_rate": [0.0159680638722554, 0.0239520958083832], "var_1_exceptions": [8, 12], "var_5_hit_rate": [0.0738522954091816, 0.0518962075848303], "var_5_exceptions": [37, 26], "target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"], "p": [1, 1], "o": [1, 1], "q": [1, 1]}, "lstm_attention_validation_results.csv": {"mse": [6.516807556152344, 6.630180835723877], "rmse": [2.5528037548065186, 2.574913740158081], "mae": [1.471358299255371, 1.489326238632202], "qlike": [1.2049862146377563, 1.2030560970306396], "var_1_hit_rate": [0.0099800399201596, 0.0119760479041916], "var_1_exceptions": [5, 6], "var_5_hit_rate": [0.0578842315369261, 0.0578842315369261], "var_5_exceptions": [29, 29], "lookback": [10, 5], "hidden_size": [16, 16], "num_layers": [1, 1], "dropout": [0.2, 0.2], "learning_rate": [0.001, 0.001], "weight_decay": [0.0, 0.0], "batch_size": [64, 64], "epochs": [80, 80], "patience": [10, 10], "target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"]}, "xgboost_validation_results.csv": {"mse": [8.571252822875977, 8.54456901550293], "rmse": [2.9276702404022217, 2.923109531402588], "mae": [1.3630386590957642, 1.3512091636657717], "qlike": [4.164594650268555, 3.840627670288086], "var_1_hit_rate": [0.1457085828343313, 0.1477045908183632], "var_1_exceptions": [73, 74], "var_5_hit_rate": [0.2035928143712574, 0.2035928143712574], "var_5_exceptions": [102, 102], "max_depth": [2, 2], "learning_rate": [0.05, 0.1], "n_estimators": [100, 50], "subsample": [0.6, 0.6], "colsample_bytree": [0.6, 0.6], "min_child_weight": [5, 1], "reg_lambda": [10, 10], "target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"]}, "msgarch_grid_search_results.csv": {"target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"], "k": [2, 2], "model": ["gjrGARCH", "gjrGARCH"], "dist": ["std", "std"], "aic": [11130.578469974882, 11115.262063731756]}, "msgarch_validation_results.csv": {"target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"], "k": [2, 2], "model": ["gjrGARCH", "gjrGARCH"], "dist": ["std", "std"], "mse": [6.9858286885496375, 6.935654521882195], "rmse": [2.64307182811017, 2.633563084849534], "mae": [1.5077571455697687, 1.509067919915724], "qlike": [1.2350648739898304, 1.2317151206910095], "var_1_hit_rate": [0.0179640718562874, 0.0179640718562874], "var_1_exceptions": [9, 9], "var_5_hit_rate": [0.0658682634730539, 0.0658682634730539], "var_5_exceptions": [33, 33]}, "transformer_validation_results.csv": {"mse": [6.729345798492432, 6.743361949920654], "rmse": [2.5940983295440674, 2.5967984199523926], "mae": [1.564149260520935, 1.512477993965149], "qlike": [1.229272484779358, 1.237881779670715], "var_1_hit_rate": [0.0039920159680638, 0.0219560878243512], "var_1_exceptions": [2, 11], "var_5_hit_rate": [0.059880239520958, 0.0638722554890219], "var_5_exceptions": [30, 32], "lookback": [22, 22], "d_model": [32, 32], "nhead": [4, 4], "num_layers": [1, 1], "dim_feedforward": [64, 64], "dropout": [0.1, 0.1], "learning_rate": [0.001, 0.001], "weight_decay": [0.0, 0.0], "batch_size": [64, 64], "epochs": [30, 30], "patience": [5, 5], "target": ["SPY", "SPY"], "freq": ["daily", "daily"], "exog": ["no_exog", "with_exog"]}}

def load_best_params(model_name: str, csv_name: str) -> pd.DataFrame:
    """Load the saved validation/best-params from disk if available, else fallback to preloaded."""
    path = MODEL_DIRS[model_name] / csv_name
    if path.exists():
        print(f"Loading {csv_name} from disk.")
        return pd.read_csv(path)
    elif csv_name in PRELOADED_CSVS:
        print(f"Loading {csv_name} from preloaded dictionary fallback.")
        return pd.DataFrame(PRELOADED_CSVS[csv_name])
    else:
        raise FileNotFoundError(
            f"{path} not found and not in preloaded dictionary — "
            f"set tune_{model_name} = True or place the CSV manually."
        )


## 5. GJR-GARCH / GARCH-X

`no_exog`: Constant-mean GJR-GARCH(p, o, q) on SPY returns.
`with_exog`: ARX-mean GJR-GARCH with cross-asset / market features as
mean-equation regressors.

Best (p, o, q) is selected by AIC on the train split.


In [35]:
from arch import arch_model

GARCH_GRID_CSV = "garch_grid_search_results.csv"
GARCH_VAL_CSV = "garch_validation_results.csv"


def garch_grid_search(target: str = TARGET, freq: str = FREQ) -> pd.DataFrame:
    """Grid search (p, o, q) for SPY/daily across both exog variants."""
    rows = []
    for exog in EXOG_VARIANTS:
        path = SPLITS_DIR / freq / exog / target / "train.csv"
        data = pd.read_csv(path, index_col="date", parse_dates=True)
        returns = data["ret"] * 100
        use_exog = exog == "with_exog"
        x_df = data[exog_columns(data)] if use_exog else None
        best_aic = np.inf; best = {}
        for p, o, q in itertools.product(range(1, 4), range(0, 2), range(1, 4)):
            try:
                if use_exog:
                    m = arch_model(returns, x=x_df, mean="ARX", lags=0,
                                   p=p, o=o, q=q, vol="Garch", dist="ged")
                else:
                    m = arch_model(returns, mean="Constant",
                                   p=p, o=o, q=q, vol="Garch", dist="ged")
                res = m.fit(disp="off")
                if res.aic < best_aic:
                    best_aic = res.aic
                    best = {"p": p, "o": o, "q": q}
            except Exception:
                continue
        rows.append({
            "target": target, "freq": freq, "exog": exog,
            "best_p": best.get("p"), "best_o": best.get("o"),
            "best_q": best.get("q"), "aic": best_aic,
        })
    df = pd.DataFrame(rows)
    df.to_csv(MODEL_DIRS["garch"] / GARCH_GRID_CSV, index=False)
    # Also persist as the validation-results CSV that the default-load
    # branch reads, with the column names downstream code expects.
    val_df = df.rename(columns={"best_p": "p", "best_o": "o",
                                 "best_q": "q"})
    val_df.to_csv(MODEL_DIRS["garch"] / GARCH_VAL_CSV, index=False)
    return df


def evaluate_garch_window(window: RFPWindow, p: int, o: int, q: int) -> tuple[dict, pd.DataFrame]:
    train_df, forecast_df = window.train, window.forecast
    train_ret = train_df["ret"] * 100
    use_exog = window.use_exog
    train_x = train_df[exog_columns(train_df)] if use_exog else None
    has_exog = use_exog and train_x is not None and len(train_x.columns) > 0

    def build(returns_s: pd.Series, x_df: pd.DataFrame | None):
        if has_exog:
            return arch_model(returns_s, x=x_df, mean="ARX", lags=0,
                              p=p, o=o, q=q, vol="Garch", dist="ged")
        return arch_model(returns_s, mean="Constant",
                          p=p, o=o, q=q, vol="Garch", dist="ged")

    try:
        res = build(train_ret, train_x).fit(disp="off")
    except Exception:
        res = None

    n_train = len(train_ret); n_fc = len(forecast_df)
    fallback = float(np.var(train_ret)) if len(train_ret) else 1.0
    pred_var_path = np.full(n_fc, fallback)
    if res is not None:
        try:
            full_ret = pd.concat([train_ret, forecast_df["ret"] * 100], ignore_index=True)
            if has_exog:
                full_x = pd.concat(
                    [train_x, forecast_df[exog_columns(forecast_df)]],
                    ignore_index=True,
                )[train_x.columns.tolist()]
            else:
                full_x = None
            fixed = build(full_ret, full_x).fix(res.params)
            sigma_path = np.asarray(fixed.conditional_volatility)
            pred_var_path = np.maximum(sigma_path[n_train:] ** 2, 1e-8)
        except Exception:
            pass

    rows = []
    for step in range(n_fc):
        pv = float(pred_var_path[step]); sigma = float(np.sqrt(pv))
        ret_pct = float(forecast_df.iloc[step]["ret"]) * 100.0
        rows.append({
            "date": forecast_df.iloc[step]["date"],
            "ret_pct": ret_pct, "realized_var": ret_pct ** 2,
            "pred_var": pv, "pred_vol": sigma,
            "VaR_1": NormalDist().inv_cdf(0.01) * sigma,
            "VaR_5": NormalDist().inv_cdf(0.05) * sigma,
        })
    fc_df = add_residual_columns(pd.DataFrame(rows))
    m = metrics(fc_df["realized_var"].values, fc_df["pred_var"].values, fc_df["ret_pct"].values)
    m.update({
        "target": window.target, "freq": FREQ,
        "exog": "with_exog" if window.use_exog else "no_exog",
        "window_id": window.window_id, "regime": window.regime,
        "n_train": window.n_train, "n_forecast": n_fc,
        "p": p, "o": o, "q": q,
    })
    return m, fc_df


# ---- run -----------------------------------------------------------------
if tune_garch:
    print("Tuning GARCH...")
    garch_grid = garch_grid_search()
else:
    garch_grid = load_best_params("garch", GARCH_VAL_CSV)
display(garch_grid)


Loading garch_grid_search_results.csv from preloaded dictionary fallback.


,target,freq,exog,best_p,best_o,best_q,aic
0,SPY,daily,no_exog,1,1,1,11126.927905
1,SPY,daily,with_exog,1,1,1,11100.087290


In [36]:
garch_results = []
garch_forecasts = {}
for exog in EXOG_VARIANTS:
    row = garch_grid[(garch_grid["target"] == TARGET) & (garch_grid["freq"] == FREQ) & (garch_grid["exog"] == exog)].iloc[0]
    # Tolerate both column schemas: grid_search CSV uses best_p/o/q,
    # validation_results CSV uses p/o/q.
    def _pick(r, *names):
        for n in names:
            if n in r and pd.notna(r[n]):
                return int(r[n])
        raise KeyError(names)
    p = _pick(row, "p", "best_p")
    o = _pick(row, "o", "best_o")
    q = _pick(row, "q", "best_q")
    print(f"GARCH {TARGET}/{FREQ}/{exog}  p={p} o={o} q={q}")
    for w in tqdm(list(iter_rfp_windows(TARGET, FREQ, use_exog=(exog == "with_exog"))),
                  desc=f"GARCH {exog}", leave=False):
        m, fc = evaluate_garch_window(w, p, o, q)
        garch_results.append(m)
        garch_forecasts[f"{TARGET}_{FREQ}_{exog}_{w.window_id}"] = fc

garch_results_df = pd.DataFrame(garch_results)
save_rfp_artifacts("garch", garch_results_df, garch_forecasts)
display_rfp(garch_results_df, "GJR-GARCH / GARCH-X");


GARCH SPY/daily/no_exog  p=1 o=1 q=1


GARCH SPY/daily/with_exog  p=1 o=1 q=1


Saved 44 window results -> GARCH/rfp

RFP results — GJR-GARCH / GARCH-X

Mean across all windows (per exog):


,exog,qlike,rmse
0,no_exog,1.2017,4.0730
1,with_exog,1.1874,4.0631



Mean by regime:


qlike              rmse          
exog       no_exog with_exog no_exog with_exog
regime                                        
CALM_17_19  0.4150    0.4068  2.6365    2.6260
COVID       1.9206    1.8779  8.4907    8.4515
ENERGY_22   1.3948    1.3853  2.5235    2.5359
GFC         2.1598    2.1436  7.2522    7.2333
OIL_CRASH   0.4444    0.4394  0.9193    0.9186

## 6. MS-GARCH (Markov-Switching GARCH)

Two-step procedure for `with_exog`:

1. Fit ARX mean (constant volatility) to extract residuals.
2. Fit MS-GARCH on the residuals.

`no_exog` skips the ARX step. Selection is by AIC over `k ∈ {1, 2}`,
`model ∈ {sGARCH, gjrGARCH}`, `dist ∈ {norm, std}`.

Requires the R package `MSGARCH` plus `rpy2`. If unavailable, the section is
skipped with a notice.


In [37]:
!R RHOME

/Library/Frameworks/R.framework/Resources


In [38]:
# Set this to the path printed by !R RHOME
os.environ['R_HOME'] = '/Library/Frameworks/R.framework/Resources'

try:
    import rpy2.robjects as robjects
    from rpy2.robjects.packages import importr
    msgarch_r = importr("MSGARCH")
    stats_r = importr("stats")
    MSGARCH_AVAILABLE = True
    print("Successfully connected to System R and loaded MSGARCH!")
except Exception as e:
    MSGARCH_AVAILABLE = False
    print(f"MS-GARCH unavailable: {e}")

Successfully connected to System R and loaded MSGARCH!


In [39]:
MSGARCH_GRID_CSV = "msgarch_grid_search_results.csv"
MSGARCH_VAL_CSV = "msgarch_validation_results.csv"


def msgarch_grid_search(target: str = TARGET, freq: str = FREQ) -> pd.DataFrame:
    rows = []
    for exog in EXOG_VARIANTS:
        path = SPLITS_DIR / freq / exog / target / "train.csv"
        data = pd.read_csv(path, index_col="date", parse_dates=True)
        returns = data["ret"] * 100
        use_exog = exog == "with_exog"
        if use_exog:
            x = data[exog_columns(data)]
            arx = arch_model(returns, x=x, mean="ARX", lags=0, vol="Constant").fit(disp="off")
            y = arx.resid.values
        else:
            y = returns.values
        y_r = robjects.FloatVector(y)
        best_aic = np.inf; best = {}
        for k, mtype, dist in itertools.product([1, 2], ["sGARCH", "gjrGARCH"], ["norm", "std"]):
            try:
                spec = msgarch_r.CreateSpec(
                    variance_spec=robjects.ListVector({"model": robjects.StrVector([mtype] * k)}),
                    distribution_spec=robjects.ListVector({"distribution": robjects.StrVector([dist] * k)}),
                    switch_spec=robjects.ListVector({"do.mix": False}),
                )
                fit = msgarch_r.FitML(spec=spec, data=y_r)
                ll = fit.rx2("loglik")[0]
                npar = len(fit.rx2("par"))
                aic = 2 * npar - 2 * ll
                if aic < best_aic:
                    best_aic = aic
                    best = {"k": k, "model": mtype, "dist": dist}
            except Exception:
                continue
        rows.append({"target": target, "freq": freq, "exog": exog,
                     "k": best.get("k", 1), "model": best.get("model", "sGARCH"),
                     "dist": best.get("dist", "norm"), "aic": best_aic})
    df = pd.DataFrame(rows)
    df.to_csv(MODEL_DIRS["msgarch"] / MSGARCH_GRID_CSV, index=False)
    # Mirror to the validation_results CSV (the default-load path)
    # so re-runs without tuning find the freshly tuned params.
    df.to_csv(MODEL_DIRS["msgarch"] / MSGARCH_VAL_CSV, index=False)
    return df


def evaluate_msgarch_window(window: RFPWindow, k: int, mtype: str, dist: str) -> tuple[dict, pd.DataFrame]:
    train_df, forecast_df = window.train, window.forecast
    use_exog = window.use_exog
    cols = exog_columns(train_df) if use_exog else []
    all_data = pd.concat([train_df, forecast_df], ignore_index=True)
    returns = all_data["ret"] * 100
    n_train = len(train_df); n_fc = len(forecast_df)

    if use_exog:
        x_full = all_data[cols]
        x_train = x_full.iloc[:n_train]
        arx = arch_model(returns.iloc[:n_train], x=x_train, mean="ARX", lags=0, vol="Constant").fit(disp="off")
        beta = arx.params.loc[cols].values
        const = float(arx.params.get("Const", 0.0))
        full_resids = returns.values - const - x_full.values @ beta
    else:
        full_resids = returns.values

    spec = msgarch_r.CreateSpec(
        variance_spec=robjects.ListVector({"model": robjects.StrVector([mtype] * k)}),
        distribution_spec=robjects.ListVector({"distribution": robjects.StrVector([dist] * k)}),
        switch_spec=robjects.ListVector({"do.mix": False}),
    )
    fit_r = msgarch_r.FitML(spec=spec, data=robjects.FloatVector(full_resids[:n_train]))

    rows = []
    for i in range(n_fc):
        cur_t = n_train + i
        r_resids = robjects.FloatVector(full_resids[:cur_t])
        fc = stats_r.predict(object=fit_r, newdata=r_resids, nahead=1)
        vol = float(fc.rx2("vol")[0])
        actual = float(returns.iloc[cur_t])
        rows.append({
            "date": all_data["date"].iloc[cur_t],
            "ret_pct": actual, "realized_var": actual ** 2,
            "pred_var": vol ** 2, "pred_vol": vol,
            "VaR_1": NormalDist().inv_cdf(0.01) * vol,
            "VaR_5": NormalDist().inv_cdf(0.05) * vol,
        })
    fc_df = add_residual_columns(pd.DataFrame(rows))
    m = metrics(fc_df["realized_var"].values, fc_df["pred_var"].values, fc_df["ret_pct"].values)
    m.update({
        "target": window.target, "freq": FREQ,
        "exog": "with_exog" if use_exog else "no_exog",
        "window_id": window.window_id, "regime": window.regime,
        "n_train": window.n_train, "n_forecast": n_fc,
        "k": k, "model": mtype, "dist": dist,
    })
    return m, fc_df


In [40]:
if MSGARCH_AVAILABLE:
    if tune_msgarch:
        print("Tuning MS-GARCH...")
        msgarch_grid = msgarch_grid_search()
    else:
        msgarch_grid = load_best_params("msgarch", MSGARCH_VAL_CSV)
    display(msgarch_grid[["target", "freq", "exog", "k", "model", "dist"]])
else:
    msgarch_grid = pd.DataFrame()


Loading msgarch_validation_results.csv from preloaded dictionary fallback.


,target,freq,exog,k,model,dist
0,SPY,daily,no_exog,2,gjrGARCH,std
1,SPY,daily,with_exog,2,gjrGARCH,std


In [41]:
msgarch_results_df = pd.DataFrame()
if MSGARCH_AVAILABLE:
    msgarch_results = []
    msgarch_forecasts = {}
    for exog in EXOG_VARIANTS:
        match = msgarch_grid[
            (msgarch_grid["target"] == TARGET) & (msgarch_grid["freq"] == FREQ)
            & (msgarch_grid["exog"] == exog)
        ]
        if match.empty:
            print(f"No MS-GARCH params for {exog}, skipping.")
            continue
        row = match.iloc[0]
        k, mtype, dist = int(row["k"]), row["model"], row["dist"]
        print(f"MS-GARCH {TARGET}/{FREQ}/{exog}  k={k} model={mtype} dist={dist}")
        for w in tqdm(list(iter_rfp_windows(TARGET, FREQ, use_exog=(exog == "with_exog"))),
                      desc=f"MS-GARCH {exog}", leave=False):
            try:
                m, fc = evaluate_msgarch_window(w, k, mtype, dist)
                msgarch_results.append(m)
                msgarch_forecasts[f"{TARGET}_{FREQ}_{exog}_{w.window_id}"] = fc
            except Exception as exc:
                print(f"  {w.window_id}: {exc}")
    msgarch_results_df = pd.DataFrame(msgarch_results)
    if not msgarch_results_df.empty:
        save_rfp_artifacts("msgarch", msgarch_results_df, msgarch_forecasts)
    display_rfp(msgarch_results_df, "MS-GARCH")
else:
    print("MS-GARCH skipped (rpy2/MSGARCH not installed).")


MS-GARCH SPY/daily/no_exog  k=2 model=gjrGARCH dist=std


MS-GARCH SPY/daily/with_exog  k=2 model=gjrGARCH dist=std


Saved 44 window results -> MSGARCH/rfp

RFP results — MS-GARCH

Mean across all windows (per exog):


,exog,qlike,rmse
0,no_exog,1.1857,4.0804
1,with_exog,1.1814,4.0810



Mean by regime:


qlike              rmse          
exog       no_exog with_exog no_exog with_exog
regime                                        
CALM_17_19  0.4194    0.4066  2.6423    2.6428
COVID       1.8665    1.8318  8.6044    8.4499
ENERGY_22   1.3647    1.3722  2.4622    2.4643
GFC         2.1511    2.1625  7.2579    7.3532
OIL_CRASH   0.4351    0.4323  0.9213    0.9191

## 7. LSTM with Attention

LSTM encoder + softmax temporal attention + Softplus head, predicting realised
variance. Hyperparameters tuned on the validation block (lowest QLIKE wins).
RFP evaluation trains from scratch on each window's training data with no
validation split, then forecasts the full window without refitting.


In [46]:
LSTM_VAL_CSV = "lstm_attention_validation_results.csv"


@dataclass(frozen=True)
class LSTMConfig:
    lookback: int = 22
    hidden_size: int = 32
    num_layers: int = 1
    dropout: float = 0.0
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 64
    epochs: int = 80
    patience: int = 10


class VolatilityLSTM(nn.Module):
    def __init__(self, n_features: int, config: LSTMConfig):
        super().__init__()
        lstm_drop = config.dropout if config.num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=config.hidden_size,
            num_layers=config.num_layers, dropout=lstm_drop, batch_first=True,
        )
        self.attention = nn.Sequential(
            nn.Linear(config.hidden_size, 1, bias=False),
            nn.Softmax(dim=1),
        )
        self.head = nn.Sequential(
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, 1),
            nn.Softplus(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        attn = self.attention(out)
        ctx = (attn * out).sum(dim=1)
        return self.head(ctx).squeeze(-1)


def torch_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def torch_device() -> torch.device:
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        return torch.device("cuda")
    return torch.device("cpu")


def fit_scaler(df: pd.DataFrame, columns: list[str]) -> StandardScaler:
    s = StandardScaler()
    s.fit(df[columns].to_numpy(dtype=np.float32))
    return s


def make_sequences(df: pd.DataFrame, columns: list[str], lookback: int,
                   scaler: StandardScaler, start_output_idx: int = 0):
    x_scaled = scaler.transform(df[columns].to_numpy(dtype=np.float32))
    y = realized_variance(df["ret"]).astype(np.float32)
    dates = df["date"].to_numpy()
    rets_pct = (df["ret"].to_numpy(dtype=np.float32) * 100.0)
    xs, ys, ds, rs = [], [], [], []
    for i in range(max(lookback - 1, start_output_idx), len(df)):
        s = i - lookback + 1
        xs.append(x_scaled[s : i + 1]); ys.append(y[i])
        ds.append(dates[i]); rs.append(rets_pct[i])
    return (np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32),
            np.asarray(ds), np.asarray(rs, dtype=np.float32))


def train_torch_model(model_cls, train_x: np.ndarray, train_y: np.ndarray,
                     val_x, val_y, config, device: torch.device, seed: int):
    torch_seed(seed)
    model = model_cls(train_x.shape[-1], config).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                            weight_decay=config.weight_decay)
    loss_fn = nn.MSELoss()
    ds = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    loader = DataLoader(ds, batch_size=config.batch_size, shuffle=True,
                        pin_memory=device.type == "cuda")
    best_state = None; best_val = math.inf; stale = 0
    for _ in range(config.epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if val_x is None or val_y is None or len(val_x) == 0:
            continue
        model.eval()
        with torch.no_grad():
            pv = model(torch.from_numpy(val_x).to(device)).cpu().numpy()
        v = float(np.mean((pv - val_y) ** 2))
        if v < best_val - 1e-8:
            best_val = v; stale = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= config.patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def torch_predict(model, x: np.ndarray, device: torch.device) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        out = model(torch.from_numpy(x).to(device)).cpu().numpy()
    return np.maximum(out, 1e-8)


In [47]:
def evaluate_seq_window(window: RFPWindow, model_cls, config, device, seed: int):
    train_df, forecast_df = window.train, window.forecast
    cols = feature_columns(train_df)
    scaler = fit_scaler(train_df, cols)
    train_x, train_y, _, _ = make_sequences(train_df, cols, config.lookback, scaler)
    model = train_torch_model(model_cls, train_x, train_y, None, None, config, device, seed)
    context_rows = train_df.tail(config.lookback - 1)
    combined = pd.concat([context_rows, forecast_df], ignore_index=True)
    fc_x, fc_y, fc_dates, fc_ret = make_sequences(
        combined, cols, config.lookback, scaler,
        start_output_idx=len(context_rows),
    )
    pred_var = torch_predict(model, fc_x, device)
    sigma = np.sqrt(pred_var)
    fc_df = add_residual_columns(pd.DataFrame({
        "date": pd.to_datetime(fc_dates).strftime("%Y-%m-%d"),
        "ret_pct": fc_ret, "realized_var": fc_y,
        "pred_var": pred_var, "pred_vol": sigma,
        "VaR_1": NormalDist().inv_cdf(0.01) * sigma,
        "VaR_5": NormalDist().inv_cdf(0.05) * sigma,
    }))
    m = metrics(fc_y, pred_var, fc_ret)
    m.update({
        "target": window.target, "freq": FREQ,
        "exog": "with_exog" if window.use_exog else "no_exog",
        "window_id": window.window_id, "regime": window.regime,
        "n_train": window.n_train, "n_forecast": len(fc_y),
    })
    return m, fc_df


def lstm_tune_cell(exog: str, device, seed: int) -> dict:
    """Minimal grid for re-tuning. Mirrors the original LSTM grid axes."""
    use_exog = exog == "with_exog"
    frames = load_cell(TARGET, FREQ, exog)
    cols = feature_columns(frames["train"])
    grid = []
    for lb, hs, dr in itertools.product([5, 10, 22], [16, 32, 64], [0.0, 0.2]):
        grid.append(LSTMConfig(lookback=lb, hidden_size=hs, dropout=dr))
    best_row = None
    for cfg in tqdm(grid, desc=f"tune lstm {exog}", leave=False):
        scaler = fit_scaler(frames["train"], cols)
        tx, ty, _, _ = make_sequences(frames["train"], cols, cfg.lookback, scaler)
        combined = pd.concat([frames["train"], frames["val"]], ignore_index=True)
        vx, vy, _, vr = make_sequences(combined, cols, cfg.lookback, scaler,
                                        start_output_idx=len(frames["train"]))
        model = train_torch_model(VolatilityLSTM, tx, ty, vx, vy, cfg, device, seed)
        pv = torch_predict(model, vx, device)
        row = metrics(vy, pv, vr); row.update(asdict(cfg))
        row.update({"target": TARGET, "freq": FREQ, "exog": exog})
        if best_row is None or row["qlike"] < best_row["qlike"]:
            best_row = row
    return best_row


In [48]:
lstm_device = torch_device()
print("LSTM device:", lstm_device)

if tune_lstm_attention:
    print("Tuning LSTM with attention...")
    rows = [lstm_tune_cell(e, lstm_device, SEED) for e in EXOG_VARIANTS]
    lstm_grid = pd.DataFrame(rows)
    lstm_grid.to_csv(MODEL_DIRS["lstm_attention"] / LSTM_VAL_CSV, index=False)
else:
    lstm_grid = load_best_params("lstm_attention", LSTM_VAL_CSV)
display(lstm_grid[[c for c in lstm_grid.columns if c in
    {"target","freq","exog","lookback","hidden_size","num_layers","dropout","learning_rate","qlike","rmse"}]])


LSTM device: cpu
Loading lstm_attention_validation_results.csv from preloaded dictionary fallback.


,rmse,qlike,lookback,hidden_size,num_layers,dropout,learning_rate,target,freq,exog
0,2.552804,1.204986,10,16,1,0.2,0.001,SPY,daily,no_exog
1,2.574914,1.203056,5,16,1,0.2,0.001,SPY,daily,with_exog


In [49]:
lstm_results = []
lstm_forecasts = {}
for exog in EXOG_VARIANTS:
    match = lstm_grid[(lstm_grid["target"] == TARGET) & (lstm_grid["freq"] == FREQ) & (lstm_grid["exog"] == exog)]
    if match.empty:
        print(f"No LSTM params for {exog}, skipping.")
        continue
    row = match.iloc[0]
    cfg = LSTMConfig(
        lookback=int(row["lookback"]), hidden_size=int(row["hidden_size"]),
        num_layers=int(row.get("num_layers", 1)),
        dropout=float(row.get("dropout", 0.0)),
        learning_rate=float(row.get("learning_rate", 1e-3)),
        weight_decay=float(row.get("weight_decay", 0.0)),
        batch_size=int(row.get("batch_size", 64)),
        epochs=int(row.get("epochs", 80)),
        patience=int(row.get("patience", 10)),
    )
    print(f"LSTM-Att {TARGET}/{FREQ}/{exog}  {cfg}")
    for w in tqdm(list(iter_rfp_windows(TARGET, FREQ, use_exog=(exog == "with_exog"))),
                  desc=f"LSTM-Att {exog}", leave=False):
        m, fc = evaluate_seq_window(w, VolatilityLSTM, cfg, lstm_device, SEED)
        lstm_results.append(m)
        lstm_forecasts[f"{TARGET}_{FREQ}_{exog}_{w.window_id}"] = fc

lstm_results_df = pd.DataFrame(lstm_results)
save_rfp_artifacts("lstm_attention", lstm_results_df, lstm_forecasts)
display_rfp(lstm_results_df, "LSTM with Attention");


LSTM-Att SPY/daily/no_exog  LSTMConfig(lookback=10, hidden_size=16, num_layers=1, dropout=0.2, learning_rate=0.001, weight_decay=0.0, batch_size=64, epochs=80, patience=10)


LSTM-Att SPY/daily/with_exog  LSTMConfig(lookback=5, hidden_size=16, num_layers=1, dropout=0.2, learning_rate=0.001, weight_decay=0.0, batch_size=64, epochs=80, patience=10)


Saved 44 window results -> lstm_attention/rfp

RFP results — LSTM with Attention

Mean across all windows (per exog):


,exog,qlike,rmse
0,no_exog,1.3330,4.2821
1,with_exog,1.1934,4.2497



Mean by regime:


qlike              rmse          
exog       no_exog with_exog no_exog with_exog
regime                                        
CALM_17_19  0.5041    0.4269  2.7243    2.6844
COVID       2.0961    1.6894  8.8222    9.1553
ENERGY_22   1.3733    1.4518  2.6332    2.4651
GFC         2.5112    2.1853  7.7742    7.6183
OIL_CRASH   0.4939    0.4634  0.9427    0.9310

## 8. Transformer

Encoder-only Transformer with sinusoidal positional encoding. Same I/O
contract as the LSTM (variance target, lookback windows, expanding RFP fits).


In [50]:
TRANSFORMER_VAL_CSV = "transformer_validation_results.csv"


@dataclass(frozen=True)
class TransformerConfig:
    lookback: int = 22
    d_model: int = 32
    nhead: int = 4
    num_layers: int = 1
    dim_feedforward: int = 64
    dropout: float = 0.1
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    batch_size: int = 64
    epochs: int = 30
    patience: int = 5


class _PosEnc(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x): return x + self.pe[:, : x.size(1)]


class VolatilityTransformer(nn.Module):
    def __init__(self, n_features: int, config: TransformerConfig):
        super().__init__()
        d = config.d_model
        if d % config.nhead != 0:
            d = ((d // config.nhead) + 1) * config.nhead
        self.proj = nn.Linear(n_features, d)
        self.pos = _PosEnc(d, max_len=max(config.lookback, 64))
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=config.num_layers)
        self.head = nn.Sequential(nn.Dropout(config.dropout), nn.Linear(d, 1), nn.Softplus())

    def forward(self, x):
        h = self.encoder(self.pos(self.proj(x)))
        return self.head(h[:, -1, :]).squeeze(-1)


def transformer_tune_cell(exog: str, device, seed: int) -> dict:
    use_exog = exog == "with_exog"
    frames = load_cell(TARGET, FREQ, exog)
    cols = feature_columns(frames["train"])
    grid = [TransformerConfig(lookback=lb, d_model=dm, nhead=4)
            for lb, dm in itertools.product([22], [32, 64])]
    best_row = None
    for cfg in tqdm(grid, desc=f"tune transformer {exog}", leave=False):
        scaler = fit_scaler(frames["train"], cols)
        tx, ty, _, _ = make_sequences(frames["train"], cols, cfg.lookback, scaler)
        combined = pd.concat([frames["train"], frames["val"]], ignore_index=True)
        vx, vy, _, vr = make_sequences(combined, cols, cfg.lookback, scaler,
                                        start_output_idx=len(frames["train"]))
        model = train_torch_model(VolatilityTransformer, tx, ty, vx, vy, cfg, device, seed)
        pv = torch_predict(model, vx, device)
        row = metrics(vy, pv, vr); row.update(asdict(cfg))
        row.update({"target": TARGET, "freq": FREQ, "exog": exog})
        if best_row is None or row["qlike"] < best_row["qlike"]:
            best_row = row
    return best_row


In [51]:
tx_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Transformer device:", tx_device)

if tune_transformer:
    print("Tuning Transformer...")
    rows = [transformer_tune_cell(e, tx_device, SEED) for e in EXOG_VARIANTS]
    tx_grid = pd.DataFrame(rows)
    tx_grid.to_csv(MODEL_DIRS["transformer"] / TRANSFORMER_VAL_CSV, index=False)
else:
    tx_grid = load_best_params("transformer", TRANSFORMER_VAL_CSV)
display(tx_grid[[c for c in tx_grid.columns if c in
    {"target","freq","exog","lookback","d_model","nhead","num_layers","dim_feedforward","dropout","learning_rate","qlike","rmse"}]])


Transformer device: cpu
Loading transformer_validation_results.csv from preloaded dictionary fallback.


,rmse,qlike,lookback,d_model,nhead,num_layers,dim_feedforward,dropout,learning_rate,target,freq,exog
0,2.594098,1.229272,22,32,4,1,64,0.1,0.001,SPY,daily,no_exog
1,2.596798,1.237882,22,32,4,1,64,0.1,0.001,SPY,daily,with_exog


In [52]:
tx_results = []
tx_forecasts = {}
for exog in EXOG_VARIANTS:
    match = tx_grid[(tx_grid["target"] == TARGET) & (tx_grid["freq"] == FREQ) & (tx_grid["exog"] == exog)]
    if match.empty:
        print(f"No Transformer params for {exog}, skipping.")
        continue
    row = match.iloc[0]
    cfg = TransformerConfig(
        lookback=int(row["lookback"]), d_model=int(row["d_model"]),
        nhead=int(row["nhead"]), num_layers=int(row["num_layers"]),
        dim_feedforward=int(row.get("dim_feedforward", 64)),
        dropout=float(row.get("dropout", 0.1)),
        learning_rate=float(row.get("learning_rate", 1e-3)),
        weight_decay=float(row.get("weight_decay", 0.0)),
        batch_size=int(row.get("batch_size", 64)),
        epochs=int(row.get("epochs", 30)),
        patience=int(row.get("patience", 5)),
    )
    print(f"Transformer {TARGET}/{FREQ}/{exog}  {cfg}")
    for w in tqdm(list(iter_rfp_windows(TARGET, FREQ, use_exog=(exog == "with_exog"))),
                  desc=f"Transformer {exog}", leave=False):
        m, fc = evaluate_seq_window(w, VolatilityTransformer, cfg, tx_device, SEED)
        tx_results.append(m)
        tx_forecasts[f"{TARGET}_{FREQ}_{exog}_{w.window_id}"] = fc

tx_results_df = pd.DataFrame(tx_results)
save_rfp_artifacts("transformer", tx_results_df, tx_forecasts)
display_rfp(tx_results_df, "Transformer");


Transformer SPY/daily/no_exog  TransformerConfig(lookback=22, d_model=32, nhead=4, num_layers=1, dim_feedforward=64, dropout=0.1, learning_rate=0.001, weight_decay=0.0, batch_size=64, epochs=30, patience=5)


Transformer SPY/daily/with_exog  TransformerConfig(lookback=22, d_model=32, nhead=4, num_layers=1, dim_feedforward=64, dropout=0.1, learning_rate=0.001, weight_decay=0.0, batch_size=64, epochs=30, patience=5)


Saved 44 window results -> transformer/rfp

RFP results — Transformer

Mean across all windows (per exog):


,exog,qlike,rmse
0,no_exog,1.4282,4.2689
1,with_exog,3.3657,4.3012



Mean by regime:


qlike              rmse          
exog       no_exog with_exog no_exog with_exog
regime                                        
CALM_17_19  0.4915    0.4980  2.7603    2.7897
COVID       2.5069   16.1774  9.0425    9.1728
ENERGY_22   1.6277    1.3909  2.5567    2.4251
GFC         2.4837    2.8855  7.6006    7.7418
OIL_CRASH   0.5025    0.6062  0.9515    0.9501

## 9. XGBoost

Gradient-boosted trees on the lagged features, fit on `log(realised_variance)`
to keep predictions strictly positive (predict-then-exponentiate).


In [31]:
XGBOOST_VAL_CSV = "xgboost_validation_results.csv"


@dataclass(frozen=True)
class XGBConfig:
    max_depth: int = 3
    learning_rate: float = 0.05
    n_estimators: int = 100
    subsample: float = 0.8
    colsample_bytree: float = 0.8
    min_child_weight: float = 1.0
    reg_lambda: float = 1.0


def _xgb_features(df: pd.DataFrame, use_exog: bool) -> list[str]:
    if use_exog:
        return feature_columns(df)
    base = ["ret_lag1", "ret_sq_lag1", "neg_ret_sq_lag1", "RV_5_lag1", "RV_10_lag1", "RV_22_lag1"]
    return [c for c in base if c in df.columns]


def fit_xgb(train_x: np.ndarray, train_y: np.ndarray, cfg: XGBConfig, seed: int) -> xgb.XGBRegressor:
    model = xgb.XGBRegressor(
        max_depth=cfg.max_depth, learning_rate=cfg.learning_rate,
        n_estimators=cfg.n_estimators, subsample=cfg.subsample,
        colsample_bytree=cfg.colsample_bytree,
        min_child_weight=cfg.min_child_weight, reg_lambda=cfg.reg_lambda,
        objective="reg:squarederror", random_state=seed, n_jobs=-1,
    )
    log_y = np.log(np.maximum(train_y, 1e-12).astype(np.float32))
    model.fit(train_x, log_y)
    return model


def predict_xgb(model, x: np.ndarray) -> np.ndarray:
    return np.maximum(np.exp(model.predict(x)), 1e-8).astype(np.float32)


def evaluate_xgb_window(window: RFPWindow, cfg: XGBConfig, seed: int):
    train_df, forecast_df = window.train, window.forecast
    cols = _xgb_features(train_df, window.use_exog)
    tx = train_df[cols].to_numpy(dtype=np.float32)
    ty = realized_variance(train_df["ret"]).astype(np.float32)
    fx = forecast_df[cols].to_numpy(dtype=np.float32)
    fy = realized_variance(forecast_df["ret"]).astype(np.float32)
    fr = (forecast_df["ret"].to_numpy(dtype=np.float32) * 100.0)
    model = fit_xgb(tx, ty, cfg, seed)
    pv = predict_xgb(model, fx)
    sigma = np.sqrt(pv)
    fc_df = add_residual_columns(pd.DataFrame({
        "date": forecast_df["date"].dt.strftime("%Y-%m-%d"),
        "ret_pct": fr, "realized_var": fy,
        "pred_var": pv, "pred_vol": sigma,
        "VaR_1": NormalDist().inv_cdf(0.01) * sigma,
        "VaR_5": NormalDist().inv_cdf(0.05) * sigma,
    }))
    m = metrics(fy, pv, fr)
    m.update({
        "target": window.target, "freq": FREQ,
        "exog": "with_exog" if window.use_exog else "no_exog",
        "window_id": window.window_id, "regime": window.regime,
        "n_train": window.n_train, "n_forecast": len(fy),
        **asdict(cfg),
    })
    return m, fc_df


def xgb_tune_cell(exog: str, seed: int) -> dict:
    use_exog = exog == "with_exog"
    frames = load_cell(TARGET, FREQ, exog)
    cols = _xgb_features(frames["train"], use_exog)
    tx = frames["train"][cols].to_numpy(dtype=np.float32)
    ty = realized_variance(frames["train"]["ret"]).astype(np.float32)
    vx = frames["val"][cols].to_numpy(dtype=np.float32)
    vy = realized_variance(frames["val"]["ret"]).astype(np.float32)
    vr = (frames["val"]["ret"].to_numpy(dtype=np.float32) * 100.0)
    grid = []
    for d, lr, n, mcw, lam, sub, col in itertools.product(
        [2, 3, 5], [0.01, 0.05, 0.1], [50, 100, 200], [1, 5], [1, 10], [0.6, 0.8], [0.6, 0.8]
    ):
        grid.append(XGBConfig(max_depth=d, learning_rate=lr, n_estimators=n,
                              subsample=sub, colsample_bytree=col,
                              min_child_weight=mcw, reg_lambda=lam))
    best_row = None
    for cfg in tqdm(grid, desc=f"tune xgb {exog}", leave=False):
        m = fit_xgb(tx, ty, cfg, seed)
        pv = predict_xgb(m, vx)
        row = metrics(vy, pv, vr); row.update(asdict(cfg))
        row.update({"target": TARGET, "freq": FREQ, "exog": exog})
        if best_row is None or row["qlike"] < best_row["qlike"]:
            best_row = row
    return best_row


In [32]:
if tune_xgboost:
    print("Tuning XGBoost...")
    rows = [xgb_tune_cell(e, SEED) for e in EXOG_VARIANTS]
    xgb_grid = pd.DataFrame(rows)
    xgb_grid.to_csv(MODEL_DIRS["xgboost"] / XGBOOST_VAL_CSV, index=False)
else:
    xgb_grid = load_best_params("xgboost", XGBOOST_VAL_CSV)
display(xgb_grid[[c for c in xgb_grid.columns if c in
    {"target","freq","exog","max_depth","learning_rate","n_estimators","subsample","colsample_bytree","min_child_weight","reg_lambda","qlike","rmse"}]])


Loading xgboost_validation_results.csv from preloaded dictionary fallback.


,rmse,qlike,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,min_child_weight,reg_lambda,target,freq,exog
0,2.92767,4.164595,2,0.05,100,0.6,0.6,5,10,SPY,daily,no_exog
1,2.92311,3.840628,2,0.10,50,0.6,0.6,1,10,SPY,daily,with_exog


In [33]:
xgb_results = []
xgb_forecasts = {}
for exog in EXOG_VARIANTS:
    match = xgb_grid[(xgb_grid["target"] == TARGET) & (xgb_grid["freq"] == FREQ) & (xgb_grid["exog"] == exog)]
    if match.empty:
        print(f"No XGBoost params for {exog}, skipping.")
        continue
    row = match.iloc[0]
    cfg = XGBConfig(
        max_depth=int(row["max_depth"]),
        learning_rate=float(row["learning_rate"]),
        n_estimators=int(row["n_estimators"]),
        subsample=float(row.get("subsample", 0.8)),
        colsample_bytree=float(row.get("colsample_bytree", 0.8)),
        min_child_weight=float(row.get("min_child_weight", 1.0)),
        reg_lambda=float(row.get("reg_lambda", 1.0)),
    )
    print(f"XGBoost {TARGET}/{FREQ}/{exog}  {cfg}")
    for w in tqdm(list(iter_rfp_windows(TARGET, FREQ, use_exog=(exog == "with_exog"))),
                  desc=f"XGBoost {exog}", leave=False):
        m, fc = evaluate_xgb_window(w, cfg, SEED)
        xgb_results.append(m)
        xgb_forecasts[f"{TARGET}_{FREQ}_{exog}_{w.window_id}"] = fc

xgb_results_df = pd.DataFrame(xgb_results)
save_rfp_artifacts("xgboost", xgb_results_df, xgb_forecasts)
display_rfp(xgb_results_df, "XGBoost");


XGBoost SPY/daily/no_exog  XGBConfig(max_depth=2, learning_rate=0.05, n_estimators=100, subsample=0.6, colsample_bytree=0.6, min_child_weight=5.0, reg_lambda=10.0)


XGBoost SPY/daily/with_exog  XGBConfig(max_depth=2, learning_rate=0.1, n_estimators=50, subsample=0.6, colsample_bytree=0.6, min_child_weight=1.0, reg_lambda=10.0)


Saved 44 window results -> xgboost/rfp

RFP results — XGBoost

Mean across all windows (per exog):


,exog,qlike,rmse
0,no_exog,4.9358,4.7047
1,with_exog,4.1985,4.6840



Mean by regime:


qlike              rmse          
exog       no_exog with_exog no_exog with_exog
regime                                        
CALM_17_19  3.3039    3.1681  3.0286    3.0256
COVID       5.8856    3.6884  9.9227    9.8693
ENERGY_22   4.4720    4.0790  2.7247    2.7040
GFC         8.5289    6.7798  8.4776    8.4388
OIL_CRASH   2.7756    3.0492  1.0610    1.0604

## 10. Combined Comparison

QLIKE & RMSE means across all daily SPY RFP windows, by model and exog
variant. Lower is better for both metrics.


In [53]:
def per_model_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    s = df.groupby("exog")[["qlike", "rmse"]].mean().round(4)
    s["model"] = name
    return s.reset_index()


pieces = []
for name, df in [
    ("GARCH", garch_results_df),
    ("MS-GARCH", msgarch_results_df),
    ("LSTM-Att", lstm_results_df),
    ("Transformer", tx_results_df),
    ("XGBoost", xgb_results_df),
]:
    s = per_model_summary(name, df)
    if not s.empty:
        pieces.append(s)

if pieces:
    combined = pd.concat(pieces, ignore_index=True)[["model", "exog", "qlike", "rmse"]]
    print("RFP averages — SPY daily, by model & exog variant")
    display(combined.pivot(index="model", columns="exog", values=["qlike", "rmse"]))
else:
    print("No results to combine.")


RFP averages — SPY daily, by model & exog variant


qlike              rmse          
exog        no_exog with_exog no_exog with_exog
model                                          
GARCH        1.2017    1.1874  4.0730    4.0631
LSTM-Att     1.3330    1.1934  4.2821    4.2497
MS-GARCH     1.1857    1.1814  4.0804    4.0810
Transformer  1.4282    3.3657  4.2689    4.3012
XGBoost      4.9358    4.1985  4.7047    4.6840

## 11. Diagnostic Plots

For every model × exog variant we produce four standard diagnostic plots
from the RFP forecast set. Forecasts from all RFP windows are concatenated
into a single chronological series before plotting. Each plot is saved
under `<model>/plots/SPY/daily/<exog>/`:

* `volatility_forecast_timeseries.png` — predicted σ̂ₜ overlaid on the
  observed |rₜ|, gives the visual "does the forecast move with the truth"
  check.
* `standardized_residuals.png` — zₜ = rₜ / σ̂ₜ, should look like white
  noise with unit variance if the variance specification is adequate.
* `acf_standardized_residuals.png` — should have no significant lags;
  significant autocorrelation here means the *mean* equation is mis-
  specified.
* `acf_squared_standardized_residuals.png` — should have no significant
  lags; significant autocorrelation here means the *variance* equation
  has not absorbed all the heteroskedasticity.

The same helper is used across all five models for a 1-to-1 visual
comparison.


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf


def _concat_forecasts(forecasts: dict[str, pd.DataFrame], exog: str) -> pd.DataFrame:
    """Concatenate the RFP forecast frames for one exog variant, sorted by date."""
    suffix = f"_{exog}_"
    parts = [df for key, df in forecasts.items() if suffix in key]
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"])
    return out.sort_values("date").reset_index(drop=True)


def make_diagnostic_plots(model_name: str, forecasts: dict[str, pd.DataFrame]) -> None:
    """Save the standard 4-plot diagnostic suite per exog variant."""
    if not forecasts:
        print(f"[{model_name}] no forecasts to plot")
        return
    base_dir = MODEL_DIRS[model_name] / "plots" / TARGET / FREQ
    for exog in EXOG_VARIANTS:
        fc = _concat_forecasts(forecasts, exog)
        if fc.empty:
            continue
        plot_dir = base_dir / exog
        plot_dir.mkdir(parents=True, exist_ok=True)

        # 1. Volatility forecast time series.
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(fc["date"], np.sqrt(fc["realized_var"]), color="#1f77b4",
                linewidth=1.0, label="Observed |return| (RFP windows)")
        ax.plot(fc["date"], fc["pred_vol"], color="#d62728",
                linewidth=1.0, label="Predicted volatility")
        ax.set_title(f"{model_name} — {TARGET} {FREQ} {exog}: observed vs predicted volatility")
        ax.set_xlabel("Date"); ax.set_ylabel("Volatility (%)")
        ax.legend(); fig.tight_layout()
        fig.savefig(plot_dir / "volatility_forecast_timeseries.png", dpi=150)
        plt.close(fig)

        # 2. Standardized residuals zₜ = rₜ / σ̂ₜ.
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(fc["date"], fc["std_resid"], color="#4c78a8", linewidth=0.8)
        ax.axhline(0.0, color="black", linewidth=0.8)
        ax.set_title(f"{model_name} — {TARGET} {FREQ} {exog}: standardized residuals")
        ax.set_xlabel("Date"); ax.set_ylabel("z_t = r_t / σ̂_t")
        fig.tight_layout()
        fig.savefig(plot_dir / "standardized_residuals.png", dpi=150)
        plt.close(fig)

        # 3 & 4. ACFs of zₜ and zₜ².
        max_lags = min(40, max(1, len(fc) // 4))
        for col, fname, title in [
            ("std_resid", "acf_standardized_residuals.png",
             "ACF of standardized residuals"),
            ("squared_std_resid", "acf_squared_standardized_residuals.png",
             "ACF of squared standardized residuals"),
        ]:
            fig, ax = plt.subplots(figsize=(12, 4))
            plot_acf(fc[col].dropna(), lags=max_lags, ax=ax)
            ax.set_title(f"{model_name} — {TARGET} {FREQ} {exog}: {title}")
            fig.tight_layout()
            fig.savefig(plot_dir / fname, dpi=150)
            plt.close(fig)

        if SHOW_PLOTS:
            print(f"[{model_name}/{exog}] saved 4 diagnostic plots -> "
                  f"{plot_dir.relative_to(ROOT)}")


In [ ]:
# Generate diagnostic plots for every model that produced forecasts.
_diag_inputs = [
    ("garch", garch_forecasts),
    ("msgarch", locals().get("msgarch_forecasts", {})),
    ("lstm_attention", lstm_forecasts),
    ("transformer", tx_forecasts),
    ("xgboost", xgb_forecasts),
]
for _name, _fcs in _diag_inputs:
    make_diagnostic_plots(_name, _fcs)


## 12. RFP Comparison Plots

Two cross-window plot families are produced per model and saved under
`<model>/rfp/plots/`:

* `ablation/SPY_daily_exog_ablation.png` — bar chart of mean QLIKE per
  RFP regime, comparing `no_exog` against `with_exog`. This is the
  cleanest visual answer to the question *"does adding the VIX/cross-
  asset features actually help during this regime?"*
* `per_window/SPY/daily/<exog>/<window_id>.png` — for each individual
  RFP window, observed vs predicted volatility over the 60-day forecast
  horizon. Useful for spotting which specific windows are hard.


In [ ]:
def make_ablation_plot(model_name: str, results_df: pd.DataFrame) -> None:
    """Bar chart: mean QLIKE per regime, no_exog vs with_exog."""
    if results_df.empty or results_df["exog"].nunique() < 2:
        return
    plot_dir = MODEL_DIRS[model_name] / "rfp" / "plots" / "ablation"
    plot_dir.mkdir(parents=True, exist_ok=True)
    pivot = (results_df.pivot_table(index="regime", columns="exog",
                                     values="qlike", aggfunc="mean")
             .sort_index())
    x = np.arange(len(pivot.index)); width = 0.35
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(x - width / 2, pivot.get("no_exog"), width,
           label="no_exog", color="#4c78a8")
    ax.bar(x + width / 2, pivot.get("with_exog"), width,
           label="with_exog", color="#f58518")
    ax.set_xticks(x); ax.set_xticklabels(pivot.index, rotation=15)
    ax.set_title(f"{model_name} {TARGET} {FREQ} — Exogenous Features Ablation by Regime",
                 fontweight="bold")
    ax.set_ylabel("Mean QLIKE (lower is better)")
    ax.legend(); fig.tight_layout()
    fname = f"{TARGET}_{FREQ}_exog_ablation.png"
    fig.savefig(plot_dir / fname, dpi=150)
    plt.close(fig)
    if SHOW_PLOTS:
        print(f"[{model_name}] ablation plot -> "
              f"{(plot_dir / fname).relative_to(ROOT)}")


def make_per_window_plots(model_name: str, forecasts: dict[str, pd.DataFrame],
                          results_df: pd.DataFrame) -> None:
    """One observed-vs-predicted plot per RFP window, grouped by exog."""
    if not forecasts:
        return
    base = MODEL_DIRS[model_name] / "rfp" / "plots" / "per_window" / TARGET / FREQ
    # Build a (window_id, exog) -> regime lookup so titles can name the regime.
    if results_df.empty:
        regime_lookup = {}
    else:
        regime_lookup = {
            (r["window_id"], r["exog"]): r["regime"]
            for r in results_df.to_dict("records")
        }
    n = 0
    for key, fc in forecasts.items():
        # Forecast keys are formatted as f"{TARGET}_{FREQ}_{exog}_{window_id}".
        prefix = f"{TARGET}_{FREQ}_"
        if not key.startswith(prefix):
            continue
        rest = key[len(prefix):]
        # rest is "<exog>_<window_id>"; exog is one of EXOG_VARIANTS.
        exog = next((e for e in EXOG_VARIANTS if rest.startswith(e + "_")), None)
        if exog is None:
            continue
        window_id = rest[len(exog) + 1:]
        regime = regime_lookup.get((window_id, exog), "")
        plot_dir = base / exog
        plot_dir.mkdir(parents=True, exist_ok=True)
        dates = pd.to_datetime(fc["date"])
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(dates, np.sqrt(fc["realized_var"]), color="#1f77b4",
                linewidth=1.2, label="Observed vol")
        ax.plot(dates, fc["pred_vol"], color="#d62728",
                linewidth=1.2, label="Predicted vol")
        title = f"{model_name} — {TARGET} {FREQ} {exog} — {window_id}"
        if regime:
            title += f" ({regime})"
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Date"); ax.set_ylabel("Volatility (%)")
        ax.legend(fontsize=9); fig.tight_layout()
        fig.savefig(plot_dir / f"{window_id}.png", dpi=150)
        plt.close(fig)
        n += 1
    if SHOW_PLOTS:
        print(f"[{model_name}] saved {n} per-window plots -> "
              f"{base.relative_to(ROOT)}")


In [ ]:
# Generate ablation + per-window plots for every model.
_rfp_inputs = [
    ("garch", garch_results_df, garch_forecasts),
    ("msgarch", msgarch_results_df, locals().get("msgarch_forecasts", {})),
    ("lstm_attention", lstm_results_df, lstm_forecasts),
    ("transformer", tx_results_df, tx_forecasts),
    ("xgboost", xgb_results_df, xgb_forecasts),
]
for _name, _res, _fcs in _rfp_inputs:
    make_ablation_plot(_name, _res)
    make_per_window_plots(_name, _fcs, _res)
